In [13]:
# VERIFY: imports/versions + thread envs
import sys, os, multiprocessing
def version_of(name, import_name=None):
    try:
        m = __import__(import_name or name)
        return getattr(m, "__version__", "unknown")
    except Exception as e:
        return f"NOT IMPORTABLE ({type(e).__name__}: {e})"

print("sys.executable:", sys.executable)
print("python:", sys.version.split()[0])
print("cpu_count:", multiprocessing.cpu_count())
print("Thread envs:", {k:os.environ.get(k) for k in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS')})
print()

pkgs = [
    ("numpy","numpy"),
    ("pandas","pandas"),
    ("scikit-learn","sklearn"),
    ("gplearn","gplearn"),
    ("xgboost","xgboost"),
    ("joblib","joblib"),
    ("pyarrow","pyarrow"),
]
for name, imp in pkgs:
    print(f"{name:12s} -> {version_of(name, imp)}")

sys.executable: /packages/apps/jupyter/2025-03-24/bin/python3.12
python: 3.12.9
cpu_count: 128
Thread envs: {'OMP_NUM_THREADS': '1', 'OPENBLAS_NUM_THREADS': '1', 'MKL_NUM_THREADS': '1'}

numpy        -> 1.26.4
pandas       -> 2.3.3
scikit-learn -> 1.2.2
gplearn      -> 0.4.2
xgboost      -> 2.1.1
joblib       -> 1.5.3
pyarrow      -> 22.0.0


In [14]:
import numpy as np
import pandas as pd
import sklearn
import gplearn
import xgboost
import joblib

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("sklearn:", sklearn.__version__)
print("gplearn:", gplearn.__version__)
print("xgboost:", xgboost.__version__)
print("joblib:", joblib.__version__)

NumPy: 1.26.4
Pandas: 2.3.3
sklearn: 1.2.2
gplearn: 0.4.2
xgboost: 2.1.1
joblib: 1.5.3


## Strategy 1: Cluster-wise Symbolic Regression
This section trains one symbolic regression model per feature cluster and measures downstream performance after appending all cluster-level symbolic features.

In [6]:
# %% [markdown]
# # Strategy 1 — Cluster-wise Symbolic Regression (Sequential)
# Thesis-consistent configuration: K=3, rich function set, sequential training

# %%
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import json, time, pickle
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.cluster import AgglomerativeClustering
from xgboost import XGBRegressor
from gplearn.genetic import SymbolicRegressor

SEED = 42
ART_ROOT = Path("artifacts") / "strategy1_sequential_k3"
ART_ROOT.mkdir(parents=True, exist_ok=True)

K_THESIS = 3
RICH_FUNCTION_SET = ("add","sub","mul","div","sin","cos")

# %%
def load_dataset(test_size=0.2, seed=SEED):
    ds = fetch_california_housing(as_frame=True)
    X = ds.data.copy()
    y = ds.target.copy()
    feat_names = list(X.columns)

    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=test_size, random_state=seed
    )

    scaler = StandardScaler().fit(Xtr.values)
    Xtr = pd.DataFrame(scaler.transform(Xtr.values), columns=feat_names)
    Xte = pd.DataFrame(scaler.transform(Xte.values), columns=feat_names)

    return Xtr, Xte, ytr.values, yte.values, feat_names

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def eval_models(Xtr, Xte, ytr, yte):
    models = {
        "LR": LinearRegression(),
        "Ridge": Ridge(alpha=1.0, random_state=SEED),
        "XGB": XGBRegressor(
            n_estimators=300,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            random_state=SEED,
            n_jobs=1,
        ),
    }
    out = {}
    for name, model in models.items():
        model.fit(Xtr, ytr)
        out[f"{name}_RMSE"] = rmse(yte, model.predict(Xte))
    return out

# %%
def make_feature_groups(X, K):
    C = np.corrcoef(X, rowvar=False)
    D = 1.0 - np.abs(C)
    np.fill_diagonal(D, 0.0)

    labels = AgglomerativeClustering(
        metric="precomputed", linkage="average", n_clusters=K
    ).fit_predict(D)

    return [np.where(labels == g)[0].tolist() for g in range(K)]

def fit_one_group(Xg, y, function_set, generations, population_size, pc, seed):
    est = SymbolicRegressor(
        function_set=function_set,
        generations=generations,
        population_size=population_size,
        parsimony_coefficient=pc,
        random_state=seed,
        n_jobs=1,
        verbose=0,
    )
    est.fit(Xg, y.ravel())
    return est

def sequential_fit_groups(Xtr, ytr, groups, function_set, generations, population_size, pc):
    sr_models = {}
    for gi, idxs in enumerate(groups):
        Xg = Xtr.iloc[:, idxs]
        sr_models[gi] = fit_one_group(
            Xg, ytr, function_set, generations, population_size, pc, SEED + gi
        )
    return sr_models

def add_sr_features(X, sr_models, groups):
    X_aug = X.copy()
    for gi, model in sr_models.items():
        X_aug[f"SR_G{gi+1}"] = model.predict(X.iloc[:, groups[gi]])
    return X_aug

# %%
def run_strategy1_one_setting(K, function_set):
    Xtr, Xte, ytr, yte, feat_names = load_dataset()
    groups = make_feature_groups(Xtr.values, K)

    t0 = time.time()

    base = eval_models(Xtr, Xte, ytr, yte)

    sr_models = sequential_fit_groups(
        Xtr, ytr, groups,
        function_set=function_set,
        generations=100,
        population_size=1500,
        pc=0.001,
    )

    Xtr_aug = add_sr_features(Xtr, sr_models, groups)
    Xte_aug = add_sr_features(Xte, sr_models, groups)

    aug = eval_models(Xtr_aug, Xte_aug, ytr, yte)

    dRMSE = {
        "LR_dRMSE": aug["LR_RMSE"] - base["LR_RMSE"],
        "Ridge_dRMSE": aug["Ridge_RMSE"] - base["Ridge_RMSE"],
        "XGB_dRMSE": aug["XGB_RMSE"] - base["XGB_RMSE"],
    }
    dRMSE["mean_dRMSE"] = np.mean(list(dRMSE.values()))

    elapsed = time.time() - t0

    return {"dRMSE": dRMSE, "elapsed_s": elapsed}

print("Strategy 1 ready (sequential, K=3, rich).")

Strategy 1 ready (sequential, K=3, rich).


In [7]:
# %% [markdown]
# ## Run Strategy 1 runtime (final)

# %%
summary = run_strategy1_one_setting(
    K=K_THESIS,
    function_set=RICH_FUNCTION_SET
)

results_df = pd.DataFrame([{
    "strategy": "Strategy 1",
    "K": K_THESIS,
    "function_set": "rich",
    "LR_dRMSE": summary["dRMSE"]["LR_dRMSE"],
    "Ridge_dRMSE": summary["dRMSE"]["Ridge_dRMSE"],
    "XGB_dRMSE": summary["dRMSE"]["XGB_dRMSE"],
    "mean_dRMSE": summary["dRMSE"]["mean_dRMSE"],
    "elapsed_s": summary["elapsed_s"],
}])

results_df

,strategy,K,function_set,LR_dRMSE,Ridge_dRMSE,XGB_dRMSE,mean_dRMSE,elapsed_s
0,Strategy 1,3,rich,-0.093222,-0.093188,-0.005279,-0.063896,687.128612


## Strategy 2: Stochastic Cluster Sampling Symbolic Regression
Thesis-consistent configuration: K=3, rich function set, single SR model per run

In [8]:
# %%
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import json
import time
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.cluster import AgglomerativeClustering
from xgboost import XGBRegressor
from gplearn.genetic import SymbolicRegressor

SEED = 42
K_THESIS = 3
RICH_FUNCTION_SET = ("add", "sub", "mul", "div", "sin", "cos")
ART_ROOT = Path("artifacts") / "strategy2_stochastic_k3_rich"
ART_ROOT.mkdir(parents=True, exist_ok=True)

In [9]:
# %%
def load_dataset(test_size=0.2, seed=SEED):
    ds = fetch_california_housing(as_frame=True)
    X = ds.data.copy()
    y = ds.target.copy()
    feat_names = list(X.columns)

    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=test_size, random_state=seed
    )

    scaler = StandardScaler().fit(Xtr.values)
    Xtr = pd.DataFrame(scaler.transform(Xtr.values), columns=feat_names)
    Xte = pd.DataFrame(scaler.transform(Xte.values), columns=feat_names)

    return Xtr, Xte, ytr.values, yte.values, feat_names

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def eval_models(Xtr, Xte, ytr, yte):
    models = {
        "LR": LinearRegression(),
        "Ridge": Ridge(alpha=1.0, random_state=SEED),
        "XGB": XGBRegressor(
            n_estimators=300,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            random_state=SEED,
            n_jobs=1,
        ),
    }
    out = {}
    for name, model in models.items():
        model.fit(Xtr, ytr)
        out[f"{name}_RMSE"] = rmse(yte, model.predict(Xte))
    return out

def make_feature_groups(X, K):
    C = np.corrcoef(X, rowvar=False)
    D = 1.0 - np.abs(C)
    np.fill_diagonal(D, 0.0)

    labels = AgglomerativeClustering(
        metric="precomputed", linkage="average", n_clusters=K
    ).fit_predict(D)

    groups = [np.where(labels == g)[0].tolist() for g in range(K)]
    return sorted(groups, key=len)

In [10]:
# %%
def sample_one_from_each_group(groups, rng):
    subset = []
    for g in groups:
        if len(g) > 0:
            subset.append(int(rng.choice(g)))
    return sorted(subset)

def fit_sr_on_subset(
    Xtr, ytr, Xte, feat_names, subset_idxs,
    function_set=RICH_FUNCTION_SET,
    generations=100,
    population_size=1500,
    parsimony=0.001,
    random_state=SEED,
):
    sub_feats = [feat_names[j] for j in subset_idxs]

    Xtr_df = pd.DataFrame(Xtr.iloc[:, subset_idxs].values, columns=sub_feats)
    Xte_df = pd.DataFrame(Xte.iloc[:, subset_idxs].values, columns=sub_feats)

    est = SymbolicRegressor(
        function_set=list(function_set),
        generations=generations,
        population_size=population_size,
        parsimony_coefficient=parsimony,
        feature_names=sub_feats,
        random_state=random_state,
        n_jobs=1,
        verbose=0,
    )

    est.fit(Xtr_df, ytr.ravel())
    tr_pred = est.predict(Xtr_df)
    te_pred = est.predict(Xte_df)

    return est, tr_pred, te_pred, sub_feats

In [11]:
# %%
def run_strategy2_one_setting(K=K_THESIS, function_set=RICH_FUNCTION_SET, seed=SEED):
    Xtr, Xte, ytr, yte, feat_names = load_dataset()
    groups = make_feature_groups(Xtr.values, K)

    t0 = time.time()
    base = eval_models(Xtr, Xte, ytr, yte)

    rng = np.random.default_rng(seed)
    subset_idxs = sample_one_from_each_group(groups, rng)

    est, tr_pred, te_pred, sub_feats = fit_sr_on_subset(
        Xtr, ytr, Xte, feat_names, subset_idxs,
        function_set=function_set,
        generations=100,
        population_size=1500,
        parsimony=0.001,
        random_state=seed,
    )

    Xtr_aug = Xtr.copy()
    Xte_aug = Xte.copy()
    Xtr_aug["SR_1"] = tr_pred
    Xte_aug["SR_1"] = te_pred

    aug = eval_models(Xtr_aug, Xte_aug, ytr, yte)

    dRMSE = {
        "LR_dRMSE": aug["LR_RMSE"] - base["LR_RMSE"],
        "Ridge_dRMSE": aug["Ridge_RMSE"] - base["Ridge_RMSE"],
        "XGB_dRMSE": aug["XGB_RMSE"] - base["XGB_RMSE"],
    }
    dRMSE["mean_dRMSE"] = float(np.mean(list(dRMSE.values())))

    run_id = f"strategy2_K{K}_rich_seed{seed}"
    run_dir = ART_ROOT / run_id
    run_dir.mkdir(parents=True, exist_ok=True)

    with open(run_dir / "sr_model.joblib", "wb") as f:
        pickle.dump(est, f)

    with open(run_dir / "summary.json", "w") as f:
        json.dump(
            {
                "run_id": run_id,
                "K": K,
                "function_set": "rich",
                "subset_features": sub_feats,
                "baseline": base,
                "augmented": aug,
                "dRMSE": dRMSE,
                "elapsed_s": time.time() - t0,
            },
            f,
            indent=2,
        )

    return {
        "run_id": run_id,
        "K": K,
        "function_set": "rich",
        "subset_features": sub_feats,
        "baseline": base,
        "augmented": aug,
        "dRMSE": dRMSE,
        "elapsed_s": time.time() - t0,
    }

print("Strategy 2 ready (sequential, K=3, rich).")

Strategy 2 ready (sequential, K=3, rich).


In [12]:
# %%
summary = run_strategy2_one_setting()

results_df = pd.DataFrame([{
    "strategy": "Strategy 2",
    "K": summary["K"],
    "function_set": summary["function_set"],
    "LR_dRMSE": summary["dRMSE"]["LR_dRMSE"],
    "Ridge_dRMSE": summary["dRMSE"]["Ridge_dRMSE"],
    "XGB_dRMSE": summary["dRMSE"]["XGB_dRMSE"],
    "mean_dRMSE": summary["dRMSE"]["mean_dRMSE"],
    "elapsed_s": summary["elapsed_s"],
    "subset_features": ", ".join(summary["subset_features"]),
}])

results_df

,strategy,K,function_set,LR_dRMSE,Ridge_dRMSE,XGB_dRMSE,mean_dRMSE,elapsed_s,subset_features
0,Strategy 2,3,rich,-0.038981,-0.038972,0.001026,-0.025642,216.237506,"AveBedrms, AveOccup, Latitude"


## Hybrid Strategy: IGS-SR
Thesis-consistent configuration: K=3, rich function set, alpha=2.0, single run

In [15]:
# %%
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import json
import time
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.cluster import AgglomerativeClustering
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
from gplearn.genetic import SymbolicRegressor

SEED = 42
K_THESIS = 3
ALPHA = 2.0
RICH_FUNCTION_SET = ("add", "sub", "mul", "div", "sin", "cos")
ART_ROOT = Path("artifacts") / "hybrid_runtime_k3_rich_alpha2"
ART_ROOT.mkdir(parents=True, exist_ok=True)

In [16]:
# %%
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def eval_models(Xtr, Xte, ytr, yte):
    models = {
        "LR": LinearRegression(),
        "Ridge": Ridge(alpha=1.0, random_state=SEED),
        "XGB": XGBRegressor(
            n_estimators=300,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            random_state=SEED,
            n_jobs=1,
        ),
    }
    out = {}
    for name, model in models.items():
        model.fit(Xtr, ytr)
        out[f"{name}_RMSE"] = rmse(yte, model.predict(Xte))
    return out

def make_feature_groups(X_df, K):
    C = np.corrcoef(X_df.values, rowvar=False)
    D = 1.0 - np.abs(C)
    np.fill_diagonal(D, 0.0)
    try:
        model = AgglomerativeClustering(metric="precomputed", linkage="average", n_clusters=K)
    except TypeError:
        model = AgglomerativeClustering(affinity="precomputed", linkage="average", n_clusters=K)
    labels = model.fit_predict(D)
    groups = [np.where(labels == g)[0].tolist() for g in range(K)]
    return sorted(groups, key=len)

def sample_one_subset(groups, importances, alpha=2.0, rng=None):
    if rng is None:
        rng = np.random.default_rng(SEED)
    subset = []
    for g in groups:
        if not g:
            continue
        w = np.asarray([importances[j] for j in g], dtype=float)
        if np.allclose(w, 0.0):
            p = np.ones(len(g)) / len(g)
        else:
            p = np.power(w + 1e-12, float(alpha))
            p = p / p.sum()
        subset.append(int(rng.choice(g, p=p)))
    return sorted(subset)

def fit_one_sr_on_subset(Xtr_df, ytr, Xte_df, subset_idx,
                         function_set=RICH_FUNCTION_SET,
                         generations=100,
                         population_size=1500,
                         parsimony=0.001,
                         random_state=SEED):
    feat_names = list(Xtr_df.columns)
    cols = [feat_names[j] for j in subset_idx]

    est = SymbolicRegressor(
        function_set=list(function_set),
        generations=generations,
        population_size=population_size,
        parsimony_coefficient=parsimony,
        random_state=random_state,
        n_jobs=1,
        verbose=0,
    )

    est.fit(Xtr_df[cols], ytr.ravel())
    tr_pred = est.predict(Xtr_df[cols])
    te_pred = est.predict(Xte_df[cols])

    return est, tr_pred, te_pred, cols

print("Hybrid runtime helpers ready.")

Hybrid runtime helpers ready.


In [17]:
# %%
# Load the canonical California Housing split used in the hybrid notebook
canon_dir = Path("artifacts") / "canonical" / "california_housing"
Xtr_df = pd.read_parquet(canon_dir / "X_train.parquet")
Xte_df = pd.read_parquet(canon_dir / "X_test.parquet")
ytr = pd.read_pickle(canon_dir / "y_train.pkl").to_numpy().ravel()
yte = pd.read_pickle(canon_dir / "y_test.pkl").to_numpy().ravel()

feat_names = list(Xtr_df.columns)
print("Loaded:", Xtr_df.shape, Xte_df.shape, ytr.shape, yte.shape)

Loaded: (16512, 8) (4128, 8) (16512,) (4128,)


In [18]:
# %%
# End-to-end hybrid runtime comparison: grouping + importances + sampling + SR + evaluation
start = time.time()

# clustering
groups = make_feature_groups(Xtr_df, K_THESIS)

# importance model
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=8,
    random_state=SEED,
    n_jobs=8,
)
rf.fit(Xtr_df, ytr)
importances = rf.feature_importances_

# baseline
base = eval_models(Xtr_df, Xte_df, ytr, yte)

# importance-guided sampling
rng = np.random.default_rng(SEED)
subset_idx = sample_one_subset(groups, importances, alpha=ALPHA, rng=rng)

# one SR model
est, tr_pred, te_pred, subset_feats = fit_one_sr_on_subset(
    Xtr_df, ytr, Xte_df, subset_idx,
    function_set=RICH_FUNCTION_SET,
    generations=100,
    population_size=1500,
    parsimony=0.001,
    random_state=SEED,
)

# augmentation + evaluation
Xtr_aug = Xtr_df.copy()
Xte_aug = Xte_df.copy()
Xtr_aug["SR_1"] = tr_pred
Xte_aug["SR_1"] = te_pred

aug = eval_models(Xtr_aug, Xte_aug, ytr, yte)

dRMSE = {
    "LR_dRMSE": aug["LR_RMSE"] - base["LR_RMSE"],
    "Ridge_dRMSE": aug["Ridge_RMSE"] - base["Ridge_RMSE"],
    "XGB_dRMSE": aug["XGB_RMSE"] - base["XGB_RMSE"],
}
dRMSE["mean_dRMSE"] = float(np.mean(list(dRMSE.values())))

elapsed_s = time.time() - start

summary = {
    "strategy": "IGS-SR",
    "K": K_THESIS,
    "alpha": ALPHA,
    "function_set": "rich",
    "subset_features": subset_feats,
    "baseline": base,
    "augmented": aug,
    "dRMSE": dRMSE,
    "elapsed_s": elapsed_s,
}

with open(ART_ROOT / "summary.json", "w") as f:
    json.dump(summary, f, indent=2)

pd.DataFrame([{
    "strategy": "IGS-SR",
    "K": K_THESIS,
    "alpha": ALPHA,
    "function_set": "rich",
    "LR_dRMSE": dRMSE["LR_dRMSE"],
    "Ridge_dRMSE": dRMSE["Ridge_dRMSE"],
    "XGB_dRMSE": dRMSE["XGB_dRMSE"],
    "mean_dRMSE": dRMSE["mean_dRMSE"],
    "elapsed_s": elapsed_s,
    "subset_features": ", ".join(subset_feats),
}])

,strategy,K,alpha,function_set,LR_dRMSE,Ridge_dRMSE,XGB_dRMSE,mean_dRMSE,elapsed_s,subset_features
0,IGS-SR,3,2.0,rich,-0.000259,-0.000256,0.002372,0.000619,179.404507,"MedInc, AveOccup, Latitude"
